In [10]:
using MAT
using JuMP
import SparseArrays as sp
using Ipopt
include("MOI_wrapper.jl")
Q = matread("/home/matt/Documents/bench/optimization/maros_meszaros_qpbenchmark/data/HS118.mat")
n = convert(Int, Q["n"])
m = convert(Int, Q["m"])
Q["A"][Q["A"] .> 1e19] .= Inf
Q["l"][Q["l"] .< -1e19] .= -Inf
Q["u"][Q["u"] .> 1e19] .= Inf
Q["A"][Q["A"] .< -1e19] .= -Inf
l_b = Q["l"][end-n+1:end]
u_b = Q["u"][end-n+1:end]
l_c = Q["l"][1:end-n]
u_c = Q["u"][1:end-n]
C = Q["A"][1:end-n, :];
n, m

(15, 32)

In [12]:
Q["P"]

15×15 SparseArrays.SparseMatrixCSC{Float64, Int64} with 15 stored entries:
 0.0002   ⋅       ⋅       ⋅       ⋅      …   ⋅       ⋅       ⋅       ⋅ 
  ⋅      0.0002   ⋅       ⋅       ⋅          ⋅       ⋅       ⋅       ⋅ 
  ⋅       ⋅      0.0003   ⋅       ⋅          ⋅       ⋅       ⋅       ⋅ 
  ⋅       ⋅       ⋅      0.0002   ⋅          ⋅       ⋅       ⋅       ⋅ 
  ⋅       ⋅       ⋅       ⋅      0.0002      ⋅       ⋅       ⋅       ⋅ 
  ⋅       ⋅       ⋅       ⋅       ⋅      …   ⋅       ⋅       ⋅       ⋅ 
  ⋅       ⋅       ⋅       ⋅       ⋅          ⋅       ⋅       ⋅       ⋅ 
  ⋅       ⋅       ⋅       ⋅       ⋅          ⋅       ⋅       ⋅       ⋅ 
  ⋅       ⋅       ⋅       ⋅       ⋅          ⋅       ⋅       ⋅       ⋅ 
  ⋅       ⋅       ⋅       ⋅       ⋅          ⋅       ⋅       ⋅       ⋅ 
  ⋅       ⋅       ⋅       ⋅       ⋅      …   ⋅       ⋅       ⋅       ⋅ 
  ⋅       ⋅       ⋅       ⋅       ⋅         0.0003   ⋅       ⋅       ⋅ 
  ⋅       ⋅       ⋅       ⋅       ⋅          ⋅      0.0002   

In [14]:
model = Model(Optimizer)
@variable(model, x[1:n])
@objective(model, MIN_SENSE, 10*(x' * Q["P"] * x + (Q["q"]' * x)[1]))
@constraint(model, l_c .<= C * x .<= u_c)
@constraint(model, l_b .<= x .<= u_b);
optimize!(model)

 0.000000e+00	-6.494260e+00	 1.000618e+02	 1.411476e+01
 5.000000e+02	 6.806956e+03	 6.868683e-01	 1.173222e+02
 1.000000e+03	 6.637034e+03	 8.641475e-01	 2.633837e+01
 1.500000e+03	 6.669508e+03	 1.576026e-02	 7.352676e-01
 2.000000e+03	 6.671440e+03	 9.364757e-03	 1.444442e+00
 2.500000e+03	 6.669247e+03	 8.926648e-03	 2.687589e-01
 3.000000e+03	 6.669605e+03	 6.697938e-04	 2.160414e-02
 3.500000e+03	 6.669651e+03	 1.233523e-04	 1.796727e-02
 4.000000e+03	 6.669623e+03	 8.666752e-05	 2.628170e-03
 4.500000e+03	 6.669626e+03	 1.343895e-05	 4.043157e-04
 5.000000e+03	 6.669627e+03	 1.491078e-06	 2.171625e-04
 5.500000e+03	 6.669626e+03	 7.672639e-07	 2.299413e-05
 6.000000e+03	 6.669627e+03	 2.179973e-07	 6.630407e-06
 6.500000e+03	 6.669627e+03	 1.730122e-08	 2.514992e-06
 7.000000e+03	 6.669627e+03	 5.716903e-09	 1.805259e-07
 7.500000e+03	 6.669627e+03	 3.179093e-09	 9.432114e-08
 8.000000e+03	 6.669627e+03	 1.897025e-10	 2.977082e-08
 8.500000e+03	 6.669627e+03	 2.347011e-11	 2.622

In [7]:
model = Model(Ipopt.Optimizer)
@variable(model, x[1:n])
@objective(model, MIN_SENSE, x' * Q["P"] * x + (Q["q"]' * x)[1])
@constraint(model, l_c .<= C * x .<= u_c)
@constraint(model, l_b .<= x .<= u_b);
optimize!(model)

This is Ipopt version 3.14.16, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:       54
Number of nonzeros in Lagrangian Hessian.............:       15

Total number of variables............................:       15
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:       32
        inequality constraints with only lower bounds:        5
   inequality constraints with lower and upper bounds:       27
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 1.00e+02 6.40e-01  -1.0 0.00e+00    -  0.00e+00 0.00e+00 